In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ---------------------------
# Dati HGD (S1–S14) come nella tabella
# ---------------------------
data = {
    "EEGNet": {
        "None": [89.38, 81.25, 97.50, 95.62, 90.00, 94.38, 91.19, 87.50, 95.62, 86.25, 76.88, 93.12, 86.25, 57.50],
        "RDWT": [87.50, 83.12, 93.75, 93.12, 90.62, 88.12, 90.57, 90.00, 95.62, 88.12, 71.88, 91.88, 90.00, 78.75],
    },
    "ShallowConvNet": {
        "None": [88.75, 87.50, 96.25, 95.62, 88.75, 91.88, 88.05, 85.62, 95.62, 85.62, 68.75, 91.88, 76.25, 78.12],
        "RDWT": [86.88, 90.00, 98.75, 96.88, 90.62, 90.62, 86.79, 80.00, 96.25, 89.38, 61.25, 93.75, 76.25, 84.38],
    },
    "MBEEG_SENet": {
        "None": [92.50, 84.38, 97.50, 96.25, 91.25, 90.62, 91.82, 89.38, 95.62, 90.62, 76.88, 93.75, 89.38, 60.62],
        "RDWT": [91.88, 91.88, 95.62, 94.38, 94.38, 91.88, 91.19, 92.50, 95.00, 86.25, 76.88, 91.88, 88.12, 81.88],
    },
    "EEGTCNet": {
        "None": [87.50, 90.62, 95.62, 90.62, 90.00, 85.62, 93.08, 87.50, 95.00, 85.00, 65.62, 93.12, 84.38, 68.75],
        "RDWT": [86.25, 90.62, 94.38, 92.50, 90.00, 88.75, 88.68, 88.12, 95.00, 88.12, 73.12, 90.62, 90.62, 63.12],
    },
}

def holm_bonferroni(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.zeros(m)
    running = 0.0
    for rank, idx in enumerate(order, start=1):
        adj_p = (m - rank + 1) * p[idx]
        running = max(running, adj_p)
        adj[idx] = running
    return np.minimum(adj, 1.0)

def wilcoxon_one_sided(a, b):
    """Wilcoxon signed-rank paired, one-sided (H1: RDWT > None)."""
    try:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox", method="auto").pvalue
    except TypeError:
        return wilcoxon(b, a, alternative="greater", zero_method="wilcox").pvalue

rows = []
for model, vals in data.items():
    a, b = vals["None"], vals["RDWT"]
    p = wilcoxon_one_sided(a, b)
    rows.append({
        "Model": model,
        "Avg_None": float(np.mean(a)),
        "Avg_RDWT": float(np.mean(b)),
        "Delta_Avg_pp": float(np.mean(b) - np.mean(a)),
        "p_wilcoxon_one_sided": float(p),
    })

df = pd.DataFrame(rows)
df["p_Holm"] = holm_bonferroni(df["p_wilcoxon_one_sided"].values)

# Risultati sintetici
pd.set_option("display.precision", 4)
print(df[["Model","Delta_Avg_pp","p_wilcoxon_one_sided","p_Holm"]])

# Snippet LaTeX (aggiungi una colonna 'p' e incolla solo sulle righe RDWT)
print("\n% LaTeX: p-value Wilcoxon (paired, one-sided; opzionale Holm in nota)")
for _, r in df.iterrows():
    print(f"% {r['Model']} RDWT: & {r['p_wilcoxon_one_sided']:.4f} \\\\")


            Model  Delta_Avg_pp  p_wilcoxon_one_sided  p_Holm
0          EEGNet        0.7579                0.6368     1.0
1  ShallowConvNet        0.2243                0.2528     1.0
2     MBEEG_SENet        1.6536                0.4169     1.0
3        EEGTCNet        0.5336                0.2969     1.0

% LaTeX: p-value Wilcoxon (paired, one-sided; opzionale Holm in nota)
% EEGNet RDWT: & 0.6368 \\
% ShallowConvNet RDWT: & 0.2528 \\
% MBEEG_SENet RDWT: & 0.4169 \\
% EEGTCNet RDWT: & 0.2969 \\
